# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

Data notes: no date column, and `device_serial` is unique per row (7000 rows, 7000 serials), so there is no time ordering or repeated entity to respect. A stratified random split is therefore appropriate. `device_serial` is an identifier and is dropped. All remaining columns (channel, device age at sale, price, self-reported daily hours at registration) are known at sale/registration, i.e. before the outcome.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.base import clone
df = pd.read_csv('data.csv')
assert df.device_serial.is_unique
print(df.shape, df.isna().sum().sum(), df.claim_within_year.mean())
df.describe()

## 1. Hold out the test set (before any modelling or exploration)

In [ ]:
y_all = df['claim_within_year']
X_all = df.drop(columns=['claim_within_year','device_serial'])
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=0)
X_train, y_train = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
print(len(X_train), len(X_test), y_train.mean(), y_test.mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: one-hot channel + scaled numerics, L2 logistic regression, C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, small grid over learning rate / depth.

Selection metric: CV average precision (PR-AUC), since only ~12.6% of rows are positive. Threshold: chosen on pooled out-of-fold predictions of the winner to maximise F1. All preprocessing sits inside the pipelines, so it is refit per fold.

In [ ]:
cat, num = ['channel'], ['device_age_years','price_eur','daily_hours']
def prep(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
cands = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for C in [0.1,1,10]:
    cands[f'logreg_C{C}'] = Pipeline([('p',prep(True)),('m',LogisticRegression(C=C,max_iter=1000))])
for lr in [0.03,0.1]:
    for d in [2,3,None]:
        cands[f'hgb_lr{lr}_d{d}'] = Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(learning_rate=lr,max_depth=d,max_iter=200,random_state=0))])
list(cands)

## 3. Cross-validation on training rows only

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=1)
oof = {}; rows = []
for name, pipe in cands.items():
    p = np.zeros(len(X_train)); fold_ap=[]
    for tr, va in skf.split(X_train, y_train):
        m = clone(pipe).fit(X_train.iloc[tr], y_train.iloc[tr])
        p[va] = m.predict_proba(X_train.iloc[va])[:,1]
        fold_ap.append(average_precision_score(y_train.iloc[va], p[va]))
    oof[name]=p
    rows.append((name, np.mean(fold_ap), np.std(fold_ap), roc_auc_score(y_train,p)))
cv = pd.DataFrame(rows, columns=['model','cv_AP','AP_sd','oof_AUC']).sort_values('cv_AP',ascending=False)
cv

## 4. Choose model and threshold (CV only)

In [ ]:
best = cv.iloc[0].model
p = oof[best]
ths = np.linspace(0.02,0.8,157)
f1s = [f1_score(y_train, p>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print(best, 'threshold', round(thr,3), 'CV F1', max(f1s))
print('baseline CV AP (prevalence):', cv.set_index('model').loc['baseline','cv_AP'])

**Decision (recorded before test scoring):** the model, hyperparameters and threshold printed above are final. The test set is scored once below with no further changes.

## 5. Refit on all training rows, score test set once

In [ ]:
final = clone(cands[best]).fit(X_train, y_train)
pt = final.predict_proba(X_test)[:,1]
base = clone(cands['baseline']).fit(X_train, y_train).predict_proba(X_test)[:,1]
pred = pt >= thr
print('TEST  AP  %.3f (baseline %.3f)' % (average_precision_score(y_test,pt), average_precision_score(y_test,base)))
print('TEST  AUC %.3f' % roc_auc_score(y_test,pt))
print('TEST  precision %.3f recall %.3f F1 %.3f at threshold %.3f' % (precision_score(y_test,pred), recall_score(y_test,pred), f1_score(y_test,pred), thr))
# uncertainty: bootstrap CI on test AP (no refitting, no selection)
rng=np.random.default_rng(0); yt=y_test.values
bs=[average_precision_score(yt[i],pt[i]) for i in (rng.integers(0,len(yt),len(yt)) for _ in range(500))]
print('AP 95%% CI: %.3f-%.3f' % tuple(np.percentile(bs,[2.5,97.5])))

## Conclusion
See the printed test numbers above: the headline is the test-set AP/AUC versus the prevalence baseline. Caveats: no timestamps, so we could not test drift over time; the random split assumes future devices resemble these.